In [1]:
import json
import os
from datetime import datetime, timedelta
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import matplotlib.pyplot as plt
from prophet import Prophet

# --- Configuration ---
plt.style.use('seaborn-v0_8')
px.defaults.template = 'plotly_white'

CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)

EIA_API_KEY = os.getenv('EIA_API_KEY')
EIA_BASE_URL = 'https://api.eia.gov/v2/electricity/'

# Codes for "Carbon-Free" energy sources
GREEN_CODES = {'SUN', 'WND', 'WAT', 'GEO', 'NUC'}

## EIA Hourly Demand Fetching


In [2]:
# --- 1. Fetch Real Hourly Demand ---
@lru_cache(maxsize=None)
def fetch_eia_hourly(region: str) -> pd.DataFrame:
    """Fetch hourly demand (MW)."""
    url = EIA_BASE_URL + 'rto/region-data/data/'
    end = datetime.utcnow()
    start = end - timedelta(days=90)
    
    params = {
        'api_key': EIA_API_KEY,
        'data[0]': 'value',
        'facets[respondent][]': region,
        'frequency': 'hourly',
        'start': start.strftime('%Y-%m-%dT%H'),
        'end': end.strftime('%Y-%m-%dT%H'),
        'sort[0][column]': 'period',
        'sort[0][direction]': 'desc',
        'length': 5000,
    }
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json().get('response', {}).get('data', [])
        df = pd.DataFrame(data)
        if not df.empty:
            df['datetime'] = pd.to_datetime(df['period'])
            df['demand_MW'] = df['value'].astype(float)
            return df.sort_values('datetime')
        return pd.DataFrame()
    except Exception as e:
        print(f"⚠️ Demand fetch failed for {region}: {e}")
        return pd.DataFrame()

# --- 2. Fetch Real Fuel Mix ---
@lru_cache(maxsize=None)
def fetch_eia_fuelmix(region: str) -> float:
    """Fetch Carbon-Free Energy % (0-100)."""
    url = EIA_BASE_URL + 'rto/fuel-type-data/data/'
    end = datetime.utcnow()
    start = end - timedelta(hours=24)
    
    params = {
        'api_key': EIA_API_KEY,
        'data[0]': 'value',
        'facets[respondent][]': region,
        'frequency': 'hourly',
        'start': start.strftime('%Y-%m-%dT%H'),
        'end': end.strftime('%Y-%m-%dT%H'),
        'length': 500
    }
    
    try:
        r = requests.get(url, params=params, timeout=30)
        data = r.json().get('response', {}).get('data', [])
        if not data: return float('nan')
            
        df = pd.DataFrame(data)
        df['value'] = df['value'].astype(float)
        
        # Get most recent hour
        latest_time = df['period'].max()
        current_mix = df[df['period'] == latest_time]
        
        total = current_mix['value'].sum()
        if total == 0: return 0.0
            
        clean = current_mix[current_mix['fueltype'].isin(GREEN_CODES)]['value'].sum()
        return (clean / total) * 100.0
    except:
        return float('nan')

# --- 3. Fetch Real Price (NEW) ---
@lru_cache(maxsize=None)
def fetch_eia_price(region: str) -> float:
    """Fetch latest Wholesale Price ($/MWh)."""
    url = EIA_BASE_URL + 'wholesale-markets-data/data/'
    end = datetime.utcnow()
    start = end - timedelta(hours=24)
    
    # Note: Mapping Regions to Hubs is complex. 
    # For this prototype, we will use a heuristic: 
    # If specific price data is missing, we fallback to a calculated proxy later.
    # This generic call attempts to get any LMP data for the region.
    params = {
        'api_key': EIA_API_KEY,
        'data[0]': 'value',
        'facets[respondent][]': region,
        'frequency': 'hourly',
        'sort[0][column]': 'period',
        'sort[0][direction]': 'desc',
        'length': 10
    }
    try:
        r = requests.get(url, params=params, timeout=5)
        data = r.json().get('response', {}).get('data', [])
        if data:
            # return average of the last few reported prices
            vals = [float(x['value']) for x in data if x['value']]
            return np.mean(vals) if vals else float('nan')
    except:
        pass
    return float('nan')

# --- 4. Fetch Weather ---
@lru_cache(maxsize=None)
def fetch_temperature(lat, lon):
    try:
        url = 'https://api.open-meteo.com/v1/forecast'
        params = {'latitude': lat, 'longitude': lon, 'daily': 'temperature_2m_mean', 'past_days': 60}
        r = requests.get(url, params=params, timeout=10)
        temps = r.json().get('daily', {}).get('temperature_2m_mean', [])
        return np.mean(temps) if temps else np.nan
    except:
        return np.nan

## Compute Datacenter Scores

In [3]:

# --- MAIN EXECUTION ---
region_coords = {
    'CAL': (36.5, -119.5), 'CAR': (35.5, -80.0), 'CENT': (38.5, -94.5),
    'FLA': (28.0, -82.0), 'MIDA': (39.0, -77.0), 'MIDW': (42.0, -89.0),
    'NE': (42.5, -72.5), 'NY': (42.9, -75.3), 'NW': (45.5, -120.5),
    'SE': (33.0, -84.0), 'SW': (36.0, -111.5), 'TEN': (36.0, -86.0),
    'TEX': (31.0, -99.0),
}

records = []
print("🚀 Fetching Real Data...")

for region, (lat, lon) in region_coords.items():
    # A. Fetch Raw Data
    df_demand = fetch_eia_hourly(region)
    raw_renew = fetch_eia_fuelmix(region)
    raw_price = fetch_eia_price(region)
    raw_temp = fetch_temperature(lat, lon)
    
    # B. Compute Derived Metrics
    if not df_demand.empty:
        raw_load = df_demand['demand_MW'].iloc[-1]
        # Volatility (Standard Deviation of last 24h)
        raw_volatility = df_demand['demand_MW'].tail(24).std()
        # Peak Forecast (Simple max of last 30 days as a proxy for capacity)
        raw_peak = df_demand['demand_MW'].max()
    else:
        raw_load, raw_volatility, raw_peak = np.nan, np.nan, np.nan

    # Fallback logic if Price API returns nothing (common for some regions)
    # We proxy price using Load Stress (High Load = High Price)
    if np.isnan(raw_price) and not np.isnan(raw_load):
        raw_price = (raw_load / 1000) * 2.5 # Rough heuristic $2.50 per GW

    records.append({
        'region': region,
        'lat': lat, 'lon': lon,
        'raw_price': raw_price,
        'raw_load': raw_load,
        'raw_volatility': raw_volatility,
        'raw_peak': raw_peak,
        'raw_renew': raw_renew,
        'raw_temp': raw_temp
    })

# --- DATAFRAME CONSTRUCTION ---
dc_df = pd.DataFrame(records)

# 1. Fill Missing Data (Mean Imputation)
dc_df = dc_df.fillna(dc_df.mean(numeric_only=True))

# 2. Normalize Columns (Store as 'n_column')
# We normalize so 0 is "Bad" and 1 is "Good"
# Price: Lower is better -> 1 - norm
# Load: Lower is better -> 1 - norm
# Volatility: Lower is better -> 1 - norm
# Temp: Lower is better -> 1 - norm
# Renewables: Higher is better -> norm

def normalize(series, invert=False):
    min_v, max_v = series.min(), series.max()
    if max_v == min_v: return 0.5
    norm = (series - min_v) / (max_v - min_v)
    return (1 - norm) if invert else norm

dc_df['n_price'] = normalize(dc_df['raw_price'], invert=True)
dc_df['n_load'] = normalize(dc_df['raw_load'], invert=True)
dc_df['n_volatility'] = normalize(dc_df['raw_volatility'], invert=True)
dc_df['n_temp'] = normalize(dc_df['raw_temp'], invert=True)
dc_df['n_renew'] = normalize(dc_df['raw_renew'], invert=False) # Higher is good

# 3. Calculate Scores
# Profitability (40%): Price, Load, Volatility
dc_df['profitability'] = (
    0.40 * dc_df['n_price'] +
    0.30 * dc_df['n_load'] +
    0.30 * dc_df['n_volatility']
)

# Sustainability (60%): Renewables, Temp
dc_df['sustainability'] = (
    0.70 * dc_df['n_renew'] + 
    0.30 * dc_df['n_temp']
)

# Final Score
dc_df['dc_score'] = 0.40 * dc_df['profitability'] + 0.60 * dc_df['sustainability']

# Sort
dc_df_final = dc_df.sort_values('dc_score', ascending=False).reset_index(drop=True)

# Save EVERYTHING
dc_df_final.to_csv('datacenter_scores_real.csv', index=False)

🚀 Fetching Real Data...


/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_49029/1253555189.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()
/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_49029/1253555189.py:40: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()
/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_49029/1253555189.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()
/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_49029/

In [4]:
dc_df_final

,region,lat,lon,raw_price,raw_load,raw_volatility,raw_peak,raw_renew,raw_temp,n_price,n_load,n_volatility,n_temp,n_renew,profitability,sustainability,dc_score
0,CAR,35.5,-80.0,53.1400,21256.0,1161.310326,35825.0,62.235115,14.804478,0.887735,0.887735,0.899840,0.460236,1.000000,0.891367,0.838071,0.859389
1,NY,42.9,-75.3,44.5450,17818.0,973.780909,20325.0,46.294596,6.670149,0.931590,0.931590,0.960022,1.000000,0.658689,0.940120,0.761082,0.832697
2,TEN,36.0,-86.0,43.1700,17268.0,851.651796,26174.0,48.673019,14.958209,0.938606,0.938606,0.999215,0.450035,0.709614,0.956789,0.631740,0.761760
3,NW,45.5,-120.5,102.7850,41114.0,1498.138120,47886.0,49.890434,9.080597,0.634428,0.634428,0.791745,0.840052,0.735681,0.681623,0.766992,0.732845
4,NE,42.5,-72.5,36.5100,14604.0,903.658433,16897.0,31.877865,8.540299,0.972588,0.972588,0.982525,0.875904,0.350004,0.975569,0.507774,0.694892
5,CENT,38.5,-94.5,77.0500,30820.0,1725.046859,48354.0,49.616232,14.440299,0.765738,0.765738,0.718926,0.484401,0.729810,0.751694,0.656187,0.694390
6,SW,36.0,-111.5,31.1375,12455.0,849.206534,20834.0,25.881175,12.680597,1.000000,1.000000,1.000000,0.601169,0.221605,1.000000,0.335474,0.601285
7,SE,33.0,-84.0,64.2350,25694.0,1597.993973,38910.0,34.138589,16.319403,0.831124,0.831124,0.759700,0.359711,0.398409,0.809697,0.386800,0.555959
8,CAL,36.5,-119.5,68.8225,27529.0,1701.052449,38450.0,27.187905,16.644776,0.807717,0.807717,0.726626,0.338120,0.249584,0.783390,0.276145,0.479043
9,MIDA,39.0,-77.0,227.1250,90850.0,2685.051468,111773.0,38.825942,12.756716,0.000000,0.000000,0.410842,0.596118,0.498773,0.123253,0.527976,0.366087
